# Urban Heat Island Prediction — End-to-End EDA and Modeling
This notebook builds a **synthetic 1,000-row Urban Heat Island (UHI) dataset**, performs clean and structured EDA, trains multiple classification models, and exports the best model for web integration.


## 1) Setup


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

import joblib
from pathlib import Path

sns.set_theme(style='whitegrid')
np.random.seed(42)


## 2) Synthetic Dataset Generation (1000 Rows)


In [ ]:
N = 1000

lst = np.clip(np.random.normal(35, 5, N), 20, 52)
ndvi = np.clip(np.random.beta(2.2, 3.5, N), 0, 1)
ndbi = np.clip(np.random.beta(3.0, 2.0, N), 0, 1)
pop_density = np.clip(np.random.normal(9000, 4000, N), 500, 25000)
building_density = np.clip(np.random.normal(0.45, 0.2, N), 0.05, 0.95)
road_density = np.clip(np.random.normal(7.0, 2.5, N), 1.0, 18.0)
humidity = np.clip(np.random.normal(58, 16, N), 15, 100)
wind_speed = np.clip(np.random.normal(10, 4, N), 0.5, 30)
rainfall = np.clip(np.random.gamma(2.2, 6.0, N), 0, 120)

heat_score = (
    0.42 * (lst - 20) / (52 - 20)
    + 0.18 * ndbi
    + 0.12 * (pop_density / 25000)
    + 0.12 * building_density
    + 0.08 * (road_density / 18)
    - 0.18 * ndvi
    - 0.05 * (wind_speed / 30)
    - 0.07 * (rainfall / 120)
    - 0.04 * (humidity / 100)
    + np.random.normal(0, 0.03, N)
)

q1, q2 = np.quantile(heat_score, [0.40, 0.75])
heat_risk_level = np.where(heat_score < q1, 0, np.where(heat_score < q2, 1, 2))

df = pd.DataFrame({
    'LST': lst,
    'NDVI': ndvi,
    'NDBI': ndbi,
    'Population_Density': pop_density,
    'Building_Density': building_density,
    'Road_Density': road_density,
    'Humidity': humidity,
    'Wind_Speed': wind_speed,
    'Rainfall': rainfall,
    'Heat_Risk_Level': heat_risk_level
})

df.head()


## 3) Basic Data Quality Checks


In [ ]:
print('Shape:', df.shape)
print('
Null values:')
print(df.isnull().sum())
print('
Target distribution:')
print(df['Heat_Risk_Level'].value_counts().sort_index())


## 4) EDA — Distributions and Relationships


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
cols = ['LST', 'NDVI', 'NDBI', 'Population_Density', 'Humidity', 'Rainfall']
for ax, col in zip(axes.flatten(), cols):
    sns.histplot(df[col], kde=True, ax=ax, color='teal')
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Heat_Risk_Level', palette='viridis')
plt.title('Heat Risk Class Distribution')
plt.xlabel('Heat Risk Level (0=Normal, 1=Moderate, 2=High)')
plt.show()


In [ ]:
plt.figure(figsize=(12,8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:
fig = px.scatter(
    df,
    x='NDVI',
    y='LST',
    color=df['Heat_Risk_Level'].map({0:'Normal',1:'Moderate',2:'High'}),
    size='Population_Density',
    title='NDVI vs LST by Heat Risk Level',
    labels={'color':'Risk Level'}
)
fig.show()


## 5) Train/Test Split and Modeling


In [ ]:
feature_cols = [
    'LST', 'NDVI', 'NDBI', 'Population_Density', 'Building_Density',
    'Road_Density', 'Humidity', 'Wind_Speed', 'Rainfall'
]
X = df[feature_cols]
y = df['Heat_Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced'),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel='rbf', C=8, gamma='scale', class_weight='balanced', random_state=42))
    ])
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(
        n_estimators=250,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='multi:softmax',
        num_class=3,
        eval_metric='mlogloss',
        random_state=42
    )

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted', zero_division=0)
    results.append([name, acc, pr, rc, f1])
    trained[name] = model

results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
results_df.sort_values('F1', ascending=False)


## 6) Best Model Analysis and Export


In [ ]:
best_model_name = results_df.sort_values('F1', ascending=False).iloc[0]['Model']
best_model = trained[best_model_name]

print(f'Best model: {best_model_name}')

best_preds = best_model.predict(X_test)
print('
Classification Report:')
print(classification_report(y_test, best_preds, digits=4))

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

Path('../models').mkdir(parents=True, exist_ok=True)
Path('../data').mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, '../models/uhi_model.pkl')
df.to_csv('../data/uhi_synthetic_dataset.csv', index=False)
print('Saved model to ../models/uhi_model.pkl')
print('Saved dataset to ../data/uhi_synthetic_dataset.csv')


## 7) Quick Inference Example


In [ ]:
sample = pd.DataFrame([{
    'LST': 41.2,
    'NDVI': 0.21,
    'NDBI': 0.78,
    'Population_Density': 13000,
    'Building_Density': 0.72,
    'Road_Density': 11.0,
    'Humidity': 50,
    'Wind_Speed': 7.0,
    'Rainfall': 6.0
}])

pred = best_model.predict(sample)[0]
label_map = {0: 'Normal', 1: 'Moderate', 2: 'High'}
print('Predicted Heat Risk:', label_map[int(pred)])
